In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model



ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C6A42F3E00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C6A5480C20>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year movie  was released")
    director:str=Field(description="it is derectored by ")
    rating:float=Field(description="the movie rating out of 10")
    explanation:str=Field(description="reason behind???")


In [3]:
model_with_structure=model.with_structured_output(Movie)

In [4]:
response=model_with_structure.invoke("Provide information about mystery south indian movies like shutter island")
response

Movie(title='Drishyam', year=2013, director='Jeethu Joseph', rating=8.5, explanation='A gripping suspense thriller where a family covers up a murder, keeping the audience guessing until the end.')

Message output alongside parsed structure


In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(..., description="Title of the movie")
    year:int=Field(..., description="This year movie  was released")
    director:str=Field(..., description="it is derectored by ")
    rating:float=Field(...,description="the movie rating out of 10")
    explanation:str=Field(..., description="reason behind???")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide information about mystery south indian movies like shutter island")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for information about mystery South Indian movies similar to "Shutter Island." Let me break this down.\n\nFirst, I need to understand what "Shutter Island" is about. It\'s a psychological thriller with elements of mystery and suspense. The user wants South Indian movies in the same vein. South Indian cinema includes Tamil, Telugu, Malayalam, and Kannada films. I should focus on mystery genres from these industries.\n\nI should recall some popular mystery movies from South India. For example, "Kerala Cafe" is a Malayalam movie known for its suspense. "Aravindan" is another one, also Malayalam, with a complex plot. Tamil movies like "Paiyaa" have mystery elements. There\'s also "Mumbai Police," a Telugu film with a gripping storyline.\n\nI need to provide details like title, year, director, rating, and explanation for each movie. Let me check the parameters required by the tool: title, year, d

In [6]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget: float|None=Field(None, description="Budget in millions USD")


structured_model=model.with_structured_output(MovieDetails) 
response=structured_model.invoke("Titanic")   
response

MovieDetails(title='Titanic', year='1997', cast=[Actor(name='Leonardo DiCaprio', role='Jack Dawson'), Actor(name='Kate Winslet', role='Rose DeWitt Bukater')], genres=['Drama', 'Romance'], budget=200.0)

TypedDict


In [7]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was revealed"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

typeDict=model.with_structured_output(MovieDict)
response=typeDict.invoke("Please provide the details of the movies avenger endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [8]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget: float|None=Field(None, description="Budget in millions USD")


structured_model=model.with_structured_output(MovieDetails) 
response=structured_model.invoke("Titanic")   
response

{'budget': 200000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jack Dawson'},
  {'name': 'Kate Winslet', 'role': 'Rose DeWitt Bukater'}],
 'genres': ['Drama', 'Romance', 'Disaster'],
 'title': 'Titanic',
 'year': '1997'}

In [9]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [12]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str =Field(description="Name of the person")
    email: str=Field(description="Email address if thr person")
    phone: str=Field(description="The phone number of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from hassam ali , hassam@example.com , 345245r"}]
})
print(result["structured_response"])

name='hassam ali' email='hassam@example.com' phone='345245r'
